In [2]:
import pandas as pd
import numpy as np

print("Ortam hazır.")

Ortam hazır.


In [3]:
df = pd.read_csv("../data/raw/TMDB_movie_dataset_v11.csv")

df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."


In [3]:
df.columns.tolist()

['id',
 'title',
 'vote_average',
 'vote_count',
 'status',
 'release_date',
 'revenue',
 'runtime',
 'adult',
 'backdrop_path',
 'budget',
 'homepage',
 'imdb_id',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'tagline',
 'genres',
 'production_companies',
 'production_countries',
 'spoken_languages',
 'keywords']

In [4]:
df.shape


(1498207, 24)

In [7]:
important_cols = [
    "id",
    "title",
    "release_date",
    "runtime",
    "original_language",
    "overview",
    "tagline",
    "genres",
    "production_countries",
    "keywords",
    "vote_average",
    "vote_count",
    "popularity"
]

df[important_cols].isnull().sum().sort_values(ascending=False)

tagline                 1288423
keywords                1134563
production_countries     740116
genres                   672185
release_date             354175
overview                 348523
title                        19
id                            0
runtime                       0
original_language             0
vote_average                  0
vote_count                    0
popularity                    0
dtype: int64

In [8]:
for col in ["overview", "genres", "keywords"]:
    print(
        col,
        "dolu oranı:",
        round(df[col].notna().mean() * 100, 2),
        "%"
    )

overview dolu oranı: 76.74 %
genres dolu oranı: 55.13 %
keywords dolu oranı: 24.27 %


In [5]:
clean_df = df[
    df["id"].notna() &
    df["title"].notna() &
    (df["title"].str.strip() != "")
].copy()

clean_df.shape

(1498184, 24)

In [6]:
clean_df["has_overview"] = (
    clean_df["overview"].notna() &
    (clean_df["overview"].fillna("").str.strip() != "")
)

clean_df["has_keywords"] = (
    clean_df["keywords"].notna() &
    (clean_df["keywords"].fillna("").str.strip() != "")
)

clean_df["has_genres"] = (
    clean_df["genres"].notna() &
    (clean_df["genres"].fillna("").str.strip() != "")
)

In [7]:
clean_df[
    ["has_overview", "has_keywords", "has_genres"]
].sum()


has_overview    1148316
has_keywords     363641
has_genres       826013
dtype: int64

In [8]:
def runtime_category(runtime):
    if pd.isna(runtime) or runtime == 0:
        return "unknown"
    elif runtime < 40:
        return "short"
    elif runtime < 60:
        return "medium_short"
    elif runtime <= 150:
        return "standard"
    elif runtime <= 240:
        return "long"
    else:
        return "very_long"

clean_df["runtime_category"] = clean_df["runtime"].apply(runtime_category)

In [9]:
clean_df["runtime_category"].value_counts()


runtime_category
standard        480096
unknown         474232
short           418267
medium_short     80436
long             36942
very_long         8211
Name: count, dtype: int64

In [10]:
clean_df["release_date"] = pd.to_datetime(
    clean_df["release_date"],
    errors="coerce"
)

clean_df["release_year"] = clean_df["release_date"].dt.year

In [11]:
clean_df["release_year"].value_counts().sort_index().tail(15)

release_year
2045.0    2
2050.0    1
2052.0    1
2055.0    1
2057.0    3
2058.0    2
2059.0    1
2060.0    1
2061.0    1
2066.0    1
2069.0    1
2074.0    1
2077.0    1
2085.0    1
2099.0    4
Name: count, dtype: int64

In [12]:
clean_df[
    clean_df["release_year"].between(2024, 2026)
]["release_year"].value_counts().sort_index()

release_year
2024.0    42622
2025.0    43022
2026.0    24174
Name: count, dtype: int64

In [13]:
today = pd.Timestamp.today().normalize()

def release_category(release_date):
    if pd.isna(release_date):
        return "unknown"
    elif release_date > today:
        return "upcoming"
    else:
        return "released"

clean_df["release_category"] = clean_df["release_date"].apply(release_category)

In [14]:
clean_df["release_category"].value_counts()

release_category
released    1142343
unknown      354158
upcoming       1683
Name: count, dtype: int64